# 🕵️ Dia 83 — Briefing do capstone e regras de engajamento

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 83* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-83.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

Rafa arrancou o primeiro post-it da parede da Operação Isca: *"Agora é a fase de **provar** coisa. Você consegue mostrar que tudo isso é exatamente o que a cooperativa te entregou?"* Ao lado, o caso de treino da Orrin Valley espera o mesmo método.

Neste notebook você vai:

1. **Montar o cofre**: `originals/` e `working/`, e o SHA-256 de cada arquivo do pacote.
2. **Provar que a verificação funciona**: o `sha256sum -c` em Python, e um arquivo adulterado de propósito.
3. **Conferir o gabarito sem abrir**: hash do `07-instructor-key.md`.
4. **Jogar o quiz das regras de engajamento**: Autorizado, Proibido ou Zona cinzenta.
5. **Abrir o cofre da Operação Isca** e o registro de decisões.

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 📦 O pacote do caso (plano A: GitHub; plano B: cópia embutida)

A próxima célula guarda uma **cópia embutida** dos arquivos do *case packet* que este dia usa. Ela é o plano B: se você estiver sem internet, ou se o GitHub estiver fora, o notebook continua funcionando.

A célula seguinte tenta o plano A: baixar cada arquivo do repositório original com `urllib`. Se baixar, compara com a cópia embutida pelo SHA-256 e avisa se o original mudou desde que este notebook foi gerado (é exatamente o raciocínio de custódia do Dia 83).

In [ ]:
# Cópia embutida (plano B). Cada valor é o texto exato do arquivo do case packet.
# Não edite: se mudar um caractere, o SHA-256 muda e a custódia quebra.
EMBUTIDO = {
"README.md": "# Capstone case packet: the Castellan remittance diversion\n\n> **SYNTHETIC TRAINING MATERIAL. EVERYTHING IN THIS FOLDER IS FICTION.**\n> Orrin Valley Millworks, Castellan Hardwood Supply, Marrowline Freight Services, Brightquay Legal, Quennmarsh, and every person, bank, registrar, hosting company and carrier named in this packet were invented for this exercise. They are not based on any real organization, person, case or client, and any resemblance is coincidental. Domains use the reserved `.example` top-level name (RFC 2606), IP addresses come from the RFC 5737 documentation ranges, AS numbers come from the RFC 5398 documentation range, and phone numbers use the 555-01XX block reserved for fiction.\n\nThis packet is the evidence for the Phase 7 capstone (Days 83 to 89). It describes one business email compromise: a lookalike vendor domain, a credential-phishing page, a taken-over mailbox, a hidden inbox rule, and a $48,612.50 payment sent to the wrong bank account. It is built so that OSINT, threat intelligence, log forensics and fraud analysis each answer part of the case and none of them answers all of it.\n\n## What is in the packet\n\n| ID | File | What it is | First used |\n|---|---|---|---|\n| P1 | [`01-engagement-letter.md`](01-engagement-letter.md) | The client's request, your authorization and scope, IT environment notes, two staff statements, and what the vendor said by phone | Day 83 |\n| P2 | [`02-suspicious-email.eml`](02-suspicious-email.eml) | The phishing email as a raw message with full headers | Day 84 |\n| P3 | [`03-related-messages.md`](03-related-messages.md) | Six related messages (M1 to M7, with M2 being P2), including the fraudulent bank-change letter and its document metadata | Day 84 |\n| P4 | [`04-vendor-site-archived.md`](04-vendor-site-archived.md) | Public artifact A: the real vendor's About page, archived November 2025 | Day 84 |\n| P5 | [`05-lookalike-site-capture.md`](05-lookalike-site-capture.md) | Public artifact B: the lookalike site's About and Privacy pages, captured 17 March 2026 | Day 84 |\n| P6 | [`06-lookup-results.md`](06-lookup-results.md) | Pre-collected WHOIS, DNS, passive DNS, certificate transparency, URL scan, favicon hash and IP enrichment results (L1 to L9) | Day 84 |\n| P7 | [`logs/mail-trace.csv`](logs/mail-trace.csv) | Mail gateway message trace, 1 to 17 March, UTC | Day 86 |\n| P8 | [`logs/mailbox-audit.csv`](logs/mailbox-audit.csv) | Sign-in and mailbox audit events for two users, 9 to 17 March, UTC | Day 86 |\n| P9 | [`logs/web-proxy.log`](logs/web-proxy.log) | Web proxy log for the AP specialist's workstation, 10 and 11 March, local time with no offset recorded | Day 86 |\n| P10 | [`07-instructor-key.md`](07-instructor-key.md) | Ground truth, planted traps and a marking guide. Do not open it until your Day 89 report is finished. | After Day 89 |\n\nCite evidence by these IDs in your notes, on your case graph, and in your report: `P8 row 2026-03-10T14:36:50Z`, `P6 L3`, `P3 M4`. If a reviewer cannot find the thing you cited in under a minute, the citation is not good enough.\n\n## Rules for working the packet\n\n1. Read [`01-engagement-letter.md`](01-engagement-letter.md) before anything else. Its scope section is binding for the whole capstone.\n2. Do not run live lookups on anything here. The domains and IPs are reserved and will return nothing, or something unrelated. [`06-lookup-results.md`](06-lookup-results.md) stands in for those lookups.\n3. Work on copies. Hash the original packet files on Day 83 and check the hashes again before you submit on Day 89.\n4. Defang indicators whenever you write them outside the packet: `castellan-hardwood[.]example`, `hxxps://cstl-docshare[.]example/view/r`, `198.51.100[.]23`.\n5. Some evidence in this packet is wrong, misleading or irrelevant on purpose, just as in a real case file. Part of the exercise is deciding which is which and saying how you decided.\n\n## How the capstone uses the packet\n\n| Day | Work | Main files |\n|---|---|---|\n| 83 | Briefing, rules of engagement, evidence register | P1, all files for hashing |\n| 84 | OSINT and threat-intel recon on the external entities | P2 to P6 |\n| 85 | Intake: the case question, scope, hypotheses, plan | P1 plus your Day 84 output |\n| 86 | Log forensics and a merged UTC timeline | P7 to P9, P2 headers |\n| 87 | Fraud and social-engineering analysis of the messages | P2, P3, P4, P5 |\n| 88 | The final case graph | Everything from Days 84 to 87 |\n| 89 | The final report at calibrated confidence | The case graph and your notes |\n",
"01-engagement-letter.md": "# Engagement letter and intake notes\n\n> SYNTHETIC TRAINING MATERIAL. Orrin Valley Millworks, Castellan Hardwood Supply, Marrowline Freight Services and every person named here are fictional. Any resemblance to a real organization or person is coincidental.\n\n## Part A: engagement letter (from the client)\n\nFrom: Owen Tran, Operations Director, Orrin Valley Millworks\nTo: You, contract investigator\nDate: Tuesday 17 March 2026\nRe: Diverted vendor payment, request for investigation\n\nOwen writes:\n\nWe are a 64-person custom millwork and cabinetry manufacturer. On Friday 13 March our weekly payment run sent $48,612.50 for invoice CHS-20417 to a bank account that our long-standing lumber supplier, Castellan Hardwood Supply, says is not theirs. We found out on Monday 16 March at about 10:15 our time, when Castellan's accounts receivable manager, Dana Whitlock, phoned to ask why the invoice was unpaid.\n\nWhat we have done so far:\n\n1. Our bank has opened a recall request on the payment. We have filed a police report (reference OV-2026-03-0412) and notified our cyber insurer.\n2. Our IT contractor reset Jordan Pike's password, signed out all of Jordan's sessions, and deleted an inbox rule they found on Jordan's mailbox. They did this on 16 March before exporting any logs. They have since exported what we are sending you.\n3. We have not contacted anyone else about this.\n\nWe would like you to answer these questions:\n\n1. How did this happen, step by step?\n2. Was Jordan's email account taken over? If so, when, from where, and what did the attacker do with it?\n3. The fraudulent email quoted Castellan's real invoice, word for word. Did that information leak from us or from Castellan?\n4. Is the same group targeting other suppliers or other staff here? Is there anything else we should block?\n5. What should we change so this does not happen again?\n\nOur insurer has asked for a written report. Please treat anything you tell us as something the insurer and possibly the police will read.\n\n## Part B: authorization and scope\n\nOrrin Valley Millworks authorizes you to:\n\n- Review every file in this case packet, including the three log exports under `logs/`.\n- Run passive research on the domains, IP addresses, email addresses and phone numbers that appear in the packet, using the pre-collected lookup results in `06-lookup-results.md`. Because every domain and IP here is a reserved documentation value, live lookups return nothing useful. The lookup file stands in for what WHOIS, passive DNS, certificate transparency and a URL scanner returned when the client's IT contractor ran them on 17 March.\n- Treat the two staff statements in Part C as your interview evidence. You cannot interview the staff again before the report is due.\n- Produce a case graph and a written report for the client, its insurer and, if the client chooses, the police.\n\nThe following are out of scope. The client did not authorize them, and several would be unlawful or unsafe in a real case:\n\n- Visiting, logging in to, or submitting anything to `cstl-docshare.example` or any other attacker-controlled site, even \"just to see the page.\"\n- Contacting the attacker by email (including the freemail Reply-To address), by phone (555-0148) or any other channel, or replying to the fraudulent thread.\n- Investigating Castellan Hardwood Supply's systems, mailboxes or staff. Castellan is a separate organization and has not engaged you. You may record what Castellan told the client and recommend that the client share indicators with Castellan.\n- Researching Dana Whitlock, Jordan Pike, Maren Beaulieu or any other named person's personal social media, home address, relatives or other personal life. You are investigating what happened to accounts and money, not the private lives of the people involved.\n- Researching Marrowline Freight Services, the real business whose brand appears on the attacker's infrastructure, beyond noting that its brand was used.\n- Attempting to identify the attacker as a named individual. Attribution to a person is a police matter. You may describe infrastructure, tactics and patterns.\n- Tracing, freezing or recovering the funds. The bank and police are handling that.\n- Any active scanning of any IP address or domain.\n\n## Part C: intake notes (taken by you on the intake call, 17 March 2026)\n\nEnvironment facts from the client's IT contractor (Priya Anand, contractor):\n\n- Email is a cloud-hosted mailbox service with a webmail interface at `mail.orrinvalley.example`. All sign-ins go through the same cloud identity service.\n- MFA rollout is 70% complete. Jordan Pike's account (`jpike`) was not yet enrolled. The controller, Maren Beaulieu (`mbeaulieu`), was enrolled in February.\n- The inbound mail gateway tags outside mail with `[EXTERNAL]` in the subject. It does not rewrite links and does not flag lookalike domains.\n- The web proxy logs in the appliance's local time. Priya is \"pretty sure\" it is set to the office time zone (US/Canada Eastern, IANA zone `America/Toronto`) but has not checked.\n- The mailbox audit export covers all events for `jpike` and `mbeaulieu` from 9 March 2026 00:00 UTC to 17 March 2026 12:00 UTC, plus failed sign-ins tenant-wide from one IP that Priya noticed. It does not cover the shared `ap@` mailbox or any other staff. Audit data before 9 March was not exported.\n- The mail gateway trace covers 1 to 17 March, filtered to messages involving `castellan`, `jpike` or `mbeaulieu`.\n- Vendor master changes are made in the ERP by the controller. ERP logs were not provided. According to Maren, she changed Castellan's bank account and remittance email address on 11 March after an email from Jordan.\n\nStatement from Jordan Pike, AP specialist (summarized):\n\n> \"I got the email from Dana on Tuesday morning, the 10th. It was in the same thread as the invoice, so I didn't think twice. I clicked the link, it asked me to sign in to see the document, and I did. Then it just went to Castellan's normal website. I figured the document had moved or something. I never sent Maren anything about banking. I didn't see any more emails from Castellan that week, which I thought was a bit odd because Dana usually sends a reminder before the due date.\"\n\nStatement from Maren Beaulieu, controller (summarized):\n\n> \"On Wednesday the 11th Jordan emailed me Castellan's bank letter and asked me to update the vendor record before Friday's run. The email said Castellan's phones were down that week, so I didn't call. The letter looked right. I made the change and replied to Jordan.\"\n\nWhat Castellan told Owen by phone on 16 March (hearsay, one call, not verified):\n\n> Dana Whitlock said she never sent a banking-change email. Castellan's IT \"checked and found nothing.\" Dana also mentioned that \"a couple of other customers\" had asked about a banking change in the past two weeks.\n",
"02-suspicious-email.eml": "X-Case-Packet-Notice: SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737). Do not look up or contact anything in this file.\nReturn-Path: <bounce-7731@castellan-hardwood.example>\nReceived: from gw01.orrinvalley.example (gw01.orrinvalley.example [192.0.2.25])\n\tby mbx01.orrinvalley.example with ESMTPS id 4f1c9a2e\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:39 +0000\nReceived: from mail.castellan-hardwood.example (vps-47.pinecrate.example [203.0.113.47])\n\tby gw01.orrinvalley.example with ESMTPS id 8b3e0d17\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:37 +0000\nReceived: from [10.8.0.6] (unknown [198.51.100.23])\n\t(Authenticated sender: ar@castellan-hardwood.example)\n\tby mail.castellan-hardwood.example (Postfix) with ESMTPSA id 2C4A1E03\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:35 +0000\nAuthentication-Results: gw01.orrinvalley.example;\n\tspf=pass (gw01.orrinvalley.example: domain of bounce-7731@castellan-hardwood.example designates 203.0.113.47 as permitted sender) smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none (message not signed);\n\tdmarc=none action=none header.from=castellan-hardwood.example\nX-OVM-Gateway-Verdict: clean; score=3.1; threshold=5.0; tags=EXTERNAL\nDate: Tue, 10 Mar 2026 10:02:31 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nMessage-ID: <20260310140231.2C4A1E03@mail.castellan-hardwood.example>\nIn-Reply-To: <20260303144702.GA11873@mail.castellanhardwood.example>\nReferences: <20260303144702.GA11873@mail.castellanhardwood.example>\nUser-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Thunderbird/115.8.0\nMIME-Version: 1.0\nContent-Type: multipart/alternative; boundary=\"------------b7Qx2LmR0v8aZ3\"\n\nThis is a multi-part message in MIME format.\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/plain; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\nHi Jordan,\n\nFollowing up on the invoice below. As part of our annual audit we have\nmoved our receivables to a new account, effective this week. Please use\nthe updated remittance details for CHS-20417 and for all future payments.\nThe signed bank letter and the vendor update form are on our secure\nportal:\n\nView remittance form (secure): https://castellanhardwood.example/remittance\n\nOur office phones are being migrated to a new system this week, so email\nis the quickest way to reach me. We would appreciate it if CHS-20417 could\ngo out in Friday's payment run, as the old account closes on the 16th.\n\nThanks so much,\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0148 | ar@castellan-hardwood.example\n\nOn Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock <dwhitlock@castellanhardwood.example> wrote:\n> Good morning,\n>\n> Please find attached invoice CHS-20417 for the February white oak and\n> hard maple order (PO 7719), total $48,612.50, net 10.\n>\n> Thank you for your business.\n>\n> Dana Whitlock\n> Accounts Receivable Manager\n> Castellan Hardwood Supply\n> Tel. 555-0112 | accounts@castellanhardwood.example\n\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/html; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\n<html><body style=\"font-family:Calibri,Arial,sans-serif;font-size:11pt\">\n<p>Hi Jordan,</p>\n<p>Following up on the invoice below. As part of our annual audit we have moved our receivables to a new account, effective this week. Please use the updated remittance details for CHS-20417 and for all future payments. The signed bank letter and the vendor update form are on our secure portal:</p>\n<p><b>View remittance form (secure):</b> <a href=\"https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&amp;t=anBpa2U%3D\">https://castellanhardwood.example/remittance</a></p>\n<p>Our office phones are being migrated to a new system this week, so email is the quickest way to reach me. We would appreciate it if CHS-20417 could go out in Friday's payment run, as the old account closes on the 16th.</p>\n<p>Thanks so much,</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0148 | ar@castellan-hardwood.example</p>\n<img src=\"https://castellanhardwood.example/img/logo.png\" width=\"160\" alt=\"Castellan Hardwood Supply\">\n<div style=\"border-left:2px solid #ccc;padding-left:8px;color:#555\">\n<p>On Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock &lt;dwhitlock@castellanhardwood.example&gt; wrote:</p>\n<p>Good morning,</p>\n<p>Please find attached invoice CHS-20417 for the February white oak and hard maple order (PO 7719), total $48,612.50, net 10.</p>\n<p>Thank you for your business.</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0112 | accounts@castellanhardwood.example</p>\n</div>\n</body></html>\n\n--------------b7Qx2LmR0v8aZ3--\n",
"03-related-messages.md": "# Related messages\n\n> SYNTHETIC TRAINING MATERIAL. All organizations, people, domains, IPs and account numbers are fictional.\n\nThe client's IT contractor pulled these messages from mailboxes on 16 and 17 March 2026. Headers are trimmed to the fields that matter; the full suspicious email is in `02-suspicious-email.eml`. Message IDs (M1 to M7) are yours to use in notes and on the case graph.\n\n## M1: the real invoice (3 March)\n\nWhere found: shared mailbox `ap@orrinvalley.example`, Inbox.\n\n```\nDate: Tue, 3 Mar 2026 09:47:02 -0500\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Invoice CHS-20417\nMessage-ID: <20260303144702.GA11873@mail.castellanhardwood.example>\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\nAttachment: CHS-20417.pdf\n```\n\n```\nGood morning,\n\nPlease find attached invoice CHS-20417 for the February white oak and\nhard maple order (PO 7719), total $48,612.50, net 10.\n\nThank you for your business.\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0112 | accounts@castellanhardwood.example\n```\n\nThe attached invoice PDF names the payee as \"Castellan Hardwood Supply Inc.\" with remittance to Vellacourt Commercial Bank, account ending 4471. It has been Castellan's account on file since 2019.\n\n## M2: the suspicious email (10 March)\n\nSee `02-suspicious-email.eml`.\n\n## M3: \"Jordan\" to the controller (11 March)\n\nWhere found: `mbeaulieu` Inbox. No copy exists in Jordan's Sent Items or Deleted Items.\n\n```\nDate: Wed, 11 Mar 2026 09:52:44 -0400\nFrom: Jordan Pike <jpike@orrinvalley.example>\nTo: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nSubject: Castellan banking update - vendor change for Friday run\nMessage-ID: <a81f3c07e2d94b6e@mbx01.orrinvalley.example>\nAttachment: Castellan_Bank_Letter_2026.pdf\n```\n\n```\nHi Maren,\n\nCastellan sent over their updated banking letter (attached) - they moved\nbanks as part of their annual audit and the old account closes on the 16th.\nCan you update the vendor master before Friday's run? They also asked that\nremittance advices go to ar@castellan-hardwood.example from now on.\n\nDana mentioned their phones are down this week while they switch systems,\nso no need to call, she said email is best.\n\nThanks!\nJordan\n```\n\nText of the attachment, `Castellan_Bank_Letter_2026.pdf`, as extracted by IT:\n\n```\nCASTELLAN HARDWOOD SUPPLY LTD.\n1400 Ferrule Road, Wrenmoor Crossing  |  Tel. 555-0148  |  ar@castellan-hardwood.example\n\n9 March 2026\n\nTo our valued customers,\n\nRE: CHANGE OF BANKING DETAILS\n\nFollowing our annual financial audit, Castellan Hardwood Supply has\ntransferred its receivables to a new banking partner. Effective immediately,\nplease remit all payments to:\n\n    Bank:            Pellbrook Savings & Trust\n    Account name:    Castellan Hardwood Supply Ltd.\n    Account ending:  8830\n\nOur previous account will be closed on 16 March 2026. Payments sent to the\nprevious account after that date will be returned and may incur late fees.\n\nPlease direct any questions to our Accounts Receivable team at\nar@castellan-hardwood.example.\n\nSincerely,\n\nDana Whitlock                          G. Castellan\nAccounts Receivable Manager            Chief Financial Officer\n```\n\nDocument metadata for the attachment, from `exiftool Castellan_Bank_Letter_2026.pdf` run by IT:\n\n```\nFile Size                       : 41 kB\nPDF Version                     : 1.7\nPage Count                      : 1\nProducer                        : LibreOffice 7.6\nCreator                         : Writer\nAuthor                          : user\nTitle                           : vendor_letter_v3\nCreate Date                     : 2026:02:26 19:12:44-05:00\nModify Date                     : 2026:03:09 22:41:03-04:00\n```\n\nSHA-256 (as recorded by the mail gateway): `5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b`\n\n## M4: the controller's reply (11 March)\n\nWhere found: `mbeaulieu` Sent Items, and Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Wed, 11 Mar 2026 11:06:12 -0400\nFrom: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: RE: Castellan banking update - vendor change for Friday run\n```\n\n```\nDone, updated in the vendor master (bank + remittance email). Please save\nthe letter to the Castellan vendor file. M.\n```\n\n## M5: the real reminder (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read. The copy in `ap@` Inbox is unread.\n\n```\nDate: Thu, 12 Mar 2026 12:05:38 -0400\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Reminder: CHS-20417 due 13 March\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\n```\n\n```\nHi Jordan,\n\nFriendly reminder that CHS-20417 ($48,612.50) is due tomorrow. Our\nbanking details are unchanged from the invoice.\n\nThanks,\nDana\nTel. 555-0112\n```\n\n## M6: the attacker's follow-up (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Thu, 12 Mar 2026 12:39:58 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none; dmarc=none header.from=castellan-hardwood.example\n```\n\n```\nHi Jordan,\n\nJust confirming the update went through on your side. Could you send the\nremittance advice as soon as the payment is released tomorrow? Our auditors\nneed it for the account transition.\n\nThanks again for your help with this,\nDana\n```\n\n## M7: the automated remittance advice (13 March)\n\nWhere found: mail gateway trace only (outbound). The ERP sends remittance advices automatically to the remittance address on the vendor record.\n\n```\nDate: Fri, 13 Mar 2026 15:02:07 -0400\nFrom: Orrin Valley Millworks Payables <erp-noreply@orrinvalley.example>\nTo: ar@castellan-hardwood.example\nSubject: Remittance advice - Orrin Valley Millworks - payment 000731\n```\n\n```\nPayment 000731 has been released.\nPayee: Castellan Hardwood Supply Ltd.   Amount: $48,612.50\nInvoices: CHS-20417\nRemitted to: Pellbrook Savings & Trust, account ending 8830\n```\n",
"04-vendor-site-archived.md": "# Artifact A: Castellan Hardwood Supply \"About us\" page (archived)\n\n> SYNTHETIC TRAINING MATERIAL. Castellan Hardwood Supply, Wrenmoor Crossing and every person named here are fictional.\n\nSource: web archive capture of `https://castellanhardwood.example/about`, captured 2 November 2025 at 16:20 UTC. The client's IT contractor saved the rendered text and the page head on 17 March 2026. This is the genuine vendor's site as it looked before the incident.\n\n## Page head (excerpt)\n\n```html\n<title>About Us | Castellan Hardwood Supply</title>\n<meta name=\"description\" content=\"Family-owned hardwood lumber yard supplying cabinetmakers and furniture builders since 1986.\">\n<link rel=\"icon\" href=\"/favicon.ico\">\n<link rel=\"stylesheet\" href=\"/css/site.css?v=2025.3\">\n```\n\n## Rendered text\n\n> **About Castellan Hardwood Supply**\n>\n> Castellan Hardwood Supply has milled and supplied North American hardwoods to cabinetmakers, millwork shops and furniture builders since 1986. We are a family-owned business, now in our third generation, operating from our yard and kiln facility at 1400 Ferrule Road in Wrenmoor Crossing.\n>\n> We stock white oak, red oak, hard and soft maple, cherry, walnut and ash in 4/4 through 12/4, with custom milling to your cut list. Most orders within our delivery region ship within five business days.\n>\n> **Our team**\n>\n> Graham Castellan, President\n> Ines Castellan-Rowe, Chief Financial Officer\n> Tobias Venn, Yard and Kiln Manager\n> Dana Whitlock, Accounts Receivable Manager\n>\n> **Contact**\n>\n> Orders: orders@castellanhardwood.example\n> Accounts: accounts@castellanhardwood.example\n> Phone: 555-0112 (Monday to Friday, 7:00 to 16:00)\n>\n> **Payments**\n>\n> For your security, Castellan Hardwood Supply will never change its banking details by email alone. If you receive a request to change how you pay us, call our office at the number above before acting on it.\n>\n> © 2025 Castellan Hardwood Supply Inc.\n",
"05-lookalike-site-capture.md": "# Artifact B: the lookalike site (captured)\n\n> SYNTHETIC TRAINING MATERIAL. All organizations, people, domains and IPs are fictional.\n\nSource: the client's IT contractor captured these pages on 17 March 2026 at 13:05 UTC from an isolated, sandboxed browser that the contractor's firm uses for this purpose. The capture is the authorized collection for this case. You do not need to visit the site, and the engagement letter forbids it.\n\n## B1: `https://castellan-hardwood.example/about`\n\nPage head (excerpt):\n\n```html\n<title>About Us | Castellan Hardwood Supply</title>\n<meta name=\"description\" content=\"Family-owned hardwood lumber yard supplying cabinetmakers and furniture builders since 1986.\">\n<link rel=\"icon\" href=\"/favicon.ico\">\n<link rel=\"stylesheet\" href=\"https://castellanhardwood.example/css/site.css?v=2024.1\">\n<img class=\"logo\" src=\"https://castellanhardwood.example/img/logo.png\" alt=\"Castellan Hardwood Supply\">\n```\n\nRendered text:\n\n> **About Castellan Hardwood Supply**\n>\n> Castellan Hardwood Supply has milled and supplied North American hardwoods to cabinetmakers, millwork shops and furniture builders since 1986. We are a family-owned business, now in our third generation, operating from our yard and kiln facility at 1400 Ferrule Road in Wrenmoor Crossing.\n>\n> We stock white oak, red oak, hard and soft maple, cherry, walnut and ash in 4/4 through 12/4, with custom milling to your cut list. Most orders within our delivery region ship within five business days.\n>\n> **Banking update**\n>\n> Following our annual audit, Castellan Hardwood Supply has moved to a new banking partner as of March 2026. Existing customers will receive updated remittance instructions from our Accounts Receivable team. Please contact ar@castellan-hardwood.example with any questions.\n>\n> **Our team**\n>\n> Graham Castellan, President\n> Tobias Venn, Yard and Kiln Manager\n> Dana Whitlock, Accounts Receivable Manager\n>\n> **Contact**\n>\n> Orders: orders@castellan-hardwood.example\n> Accounts: ar@castellan-hardwood.example\n> Phone: 555-0148 (Monday to Friday, 7:00 to 16:00)\n>\n> © 2024 Castellan Hardwood Supply Inc.\n\n## B2: `https://castellan-hardwood.example/privacy`\n\nRendered text (complete):\n\n> **Privacy policy**\n>\n> Castellan Hardwood Supply collects only the information needed to process your orders and payments. We do not sell your information to third parties. Marrowline Freight Services respects your privacy and will only use your contact details to respond to your enquiry. For questions about this policy, contact ar@castellan-hardwood.example.\n\n## B3: other paths requested during the capture\n\n| Path | Status | Notes |\n|---|---|---|\n| `/` | 200 | Home page. Same layout as the real site; the hero image is hotlinked from `castellanhardwood.example`. |\n| `/contact` | 200 | Contact form. The form posts to `/assets/php/contact.php`. |\n| `/remittance` | 404 | No page at the path shown in the phishing email's link text. |\n| `/favicon.ico` | 200 | Byte-identical to the real site's favicon (see the hash in `06-lookup-results.md`). |\n",
"06-lookup-results.md": "# Pre-collected lookup results\n\n> SYNTHETIC TRAINING MATERIAL. Every domain uses the reserved `.example` name, every IP is an RFC 5737 documentation address, and every AS number is an RFC 5398 documentation ASN. Registrars, hosting companies and carriers named here are fictional.\n\nThe client's IT contractor ran these passive lookups on 17 March 2026 between 12:40 and 13:30 UTC, using WHOIS, a passive DNS service, a certificate transparency search, a URL scanning service and a favicon-hash search. In a real case you would run them yourself and log each one. Here, reading a section counts as running that lookup: log it in your collection log with the section number as the source and 17 March 2026 as the access date.\n\nSection numbers (L1 to L9) are for citation in your notes and case graph.\n\n## L1: WHOIS\n\n| Field | castellanhardwood.example | castellan-hardwood.example | cstl-docshare.example | marrowline-docs.example | quennmarsh-accounts.example |\n|---|---|---|---|---|---|\n| Created | 2004-06-14 | 2026-02-24T21:14:09Z | 2026-02-26T03:40:51Z | 2026-02-19T22:05:33Z | 2026-03-14T02:51:17Z |\n| Registrar | Beaconhill Domains | Quillfeather Names | Quillfeather Names | Quillfeather Names | Quillfeather Names |\n| Registrant org | Castellan Hardwood Supply Inc. | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY |\n| Name servers | ns1/ns2.cedarlinedns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example |\n| Expires | 2027-06-14 | 2027-02-24 | 2027-02-26 | 2027-02-19 | 2027-03-14 |\n\nContext from the registrar and DNS host's own public pages: Quillfeather Names is a low-cost registrar with about 3 million domains under management. Hollowdns is a free DNS hosting service with about 90,000 zones.\n\n## L2: current DNS records (17 March 2026)\n\n```\ncastellanhardwood.example.          A      192.0.2.140\ncastellanhardwood.example.          MX     10 mail.castellanhardwood.example.\nmail.castellanhardwood.example.     A      198.51.100.180\ncastellanhardwood.example.          TXT    \"v=spf1 ip4:198.51.100.180 -all\"\n_dmarc.castellanhardwood.example.   TXT    \"v=DMARC1; p=quarantine; rua=mailto:dmarc@castellanhardwood.example\"\n\ncastellan-hardwood.example.         A      203.0.113.47\ncastellan-hardwood.example.         MX     10 mail.castellan-hardwood.example.\nmail.castellan-hardwood.example.    A      203.0.113.47\ncastellan-hardwood.example.         TXT    \"v=spf1 ip4:203.0.113.47 -all\"\n_dmarc.castellan-hardwood.example.  NXDOMAIN\n\ncstl-docshare.example.              A      203.0.113.88\nwww.cstl-docshare.example.          CNAME  cstl-docshare.example.\n\nmarrowline-docs.example.            A      203.0.113.91\nportal.marrowline-docs.example.     A      203.0.113.91\nmail.marrowline-docs.example.       NXDOMAIN\n\nquennmarsh-accounts.example.        A      203.0.113.91\n```\n\n## L3: passive DNS history\n\n| rrname | Type | rdata | First seen (UTC) | Last seen (UTC) |\n|---|---|---|---|---|\n| castellanhardwood.example | A | 192.0.2.140 | 2019-04-02 | 2026-03-17 |\n| mail.castellanhardwood.example | A | 198.51.100.170 | 2017-08-11 | 2026-01-19 |\n| mail.castellanhardwood.example | A | 198.51.100.180 | 2026-01-19 | 2026-03-17 |\n| castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| mail.castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| cstl-docshare.example | A | 203.0.113.88 | 2026-02-27 | 2026-03-17 |\n| marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| portal.marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| mail.marrowline-docs.example | A | 198.51.100.23 | 2026-02-20 | 2026-03-09 |\n| quennmarsh-accounts.example | A | 203.0.113.91 | 2026-03-15 | 2026-03-17 |\n\nReverse lookups on the same service:\n\n| IP | Hostnames seen resolving to it | PTR record |\n|---|---|---|\n| 198.51.100.23 | mail.marrowline-docs.example (2026-02-20 to 2026-03-09) | vps-23.pinecrate.example |\n| 203.0.113.47 | castellan-hardwood.example, mail.castellan-hardwood.example | vps-47.pinecrate.example |\n| 203.0.113.91 | marrowline-docs.example, portal.marrowline-docs.example, quennmarsh-accounts.example | none |\n\n## L4: certificate transparency\n\n| Log entry ID | Names on certificate | Issuer | Not before (UTC) |\n|---|---|---|---|\n| 9048820415 | marrowline-docs.example, portal.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:37 |\n| 9048820911 | mail.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:39 |\n| 9051182231 | castellan-hardwood.example, www.castellan-hardwood.example, mail.castellan-hardwood.example | Let's Encrypt | 2026-02-25 06:02 |\n| 9051497710 | cstl-docshare.example, www.cstl-docshare.example | Let's Encrypt | 2026-02-27 04:11 |\n| 9063310054 | quennmarsh-accounts.example | Let's Encrypt | 2026-03-15 00:22 |\n| (routine) | castellanhardwood.example, www.castellanhardwood.example | Let's Encrypt | Renewed every 60 to 90 days since 2019; latest 2025-12-30 |\n\n## L5: co-hosting (reverse IP)\n\n| IP | Hosting | Domains currently hosted | Sample |\n|---|---|---|---|\n| 203.0.113.47 | Pinecrate VPS, single-tenant virtual server | 2 hostnames | See L3 |\n| 203.0.113.88 | Pinecrest Shared Hosting, shared web server | 1,412 domains | ferncastle-bakery.example, oakhollow-dental.example, rivermint-yoga.example, tidepost-news.example, cstl-docshare.example, (1,407 more) |\n| 203.0.113.91 | Driftwood Cloud VPS, single-tenant virtual server | 3 hostnames | See L3 |\n\n## L6: URL scan results\n\nS1. Scanned by the client's IT contractor, 17 March 2026 13:20 UTC, sandboxed scanner.\n\n```\nURL:         https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&t=anBpa2U%3D\nFinal URL:   (same)            Status: 200        IP: 203.0.113.88\nPage title:  Secure Document Viewer\nPage text:   \"Castellan Hardwood Supply shared a secure document with you.\n              Sign in with your work email to view Remittance_Form.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://castellanhardwood.example/img/logo.png   (third-party image load)\nForm:        POST /view/auth   fields: email (prefilled \"jpike@orrinvalley.example\"), password\nNotes:       https://cstl-docshare.example/ returns 404. Scanner followed no redirect.\n```\n\nS2. From a public scan database. Submitted by an unknown third party on 22 February 2026 09:14 UTC.\n\n```\nURL:         https://portal.marrowline-docs.example/view/r?u=[redacted by database]\nStatus:      200        IP: 203.0.113.91\nPage title:  Secure Document Viewer\nPage text:   \"Marrowline Freight Services shared a secure document with you.\n              Sign in with your work email to view Updated_Payment_Instructions.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://marrowlinefreight.example/images/brand.png   (third-party image load)\nForm:        POST /view/auth   fields: email, password\n```\n\nS3. From a public scan database. Submitted by an unknown third party on 8 October 2025 16:02 UTC.\n\n```\nURL:         https://files.brightquay-legal.example/\nStatus:      200        IP: 192.0.2.201\nPage title:  Secure Document Viewer | Brightquay Legal\nResources:   /static/viewer.js             sha256 3b7f0e92c4d15a8e6f20b9c7d41e5a3f8c62d09e1b4a7f35e8d0c6b2a91f4e07\n             /favicon.ico                  mmh3 -1742093313\nForm:        POST /sso/login\nPage footer: \"Built on the OpenDocViewer template\"\nCert history: certificates for files.brightquay-legal.example continuously since 2021\n```\n\n## L7: favicon hash search\n\n| mmh3 hash | Hosts returned |\n|---|---|\n| -1742093313 | cstl-docshare.example, portal.marrowline-docs.example, files.brightquay-legal.example |\n| 884215106 | castellanhardwood.example, castellan-hardwood.example |\n\n## L8: IP enrichment\n\n| IP | ASN | Network owner | Network type | Location (as reported by the geolocation service) | Other context |\n|---|---|---|---|---|---|\n| 192.0.2.10 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley office NAT address (confirmed by IT) |\n| 192.0.2.25 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley mail gateway gw01 (confirmed by IT) |\n| 192.0.2.140 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Castellan's real website |\n| 192.0.2.201 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Brightquay Legal document portal |\n| 198.51.100.23 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 198.51.100.77 | AS64508 | Skerrow Mobile | Mobile carrier, carrier-grade NAT | Client's metro area | Shared by many mobile subscribers |\n| 198.51.100.170 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's previous mail server |\n| 198.51.100.180 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's current mail server |\n| 203.0.113.47 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 203.0.113.88 | AS64505 | Pinecrest Shared Hosting | Shared web hosting | North America | 1,412 co-hosted domains (L5) |\n| 203.0.113.91 | AS64511 | Driftwood Cloud | Datacenter / VPS | North America | None |\n| 203.0.113.200 | AS64507 | Keelson Hosting | Datacenter | Asia-Pacific datacenter | Listed on a public blocklist for password spraying against cloud mail tenants since January 2026 |\n\nGeolocation for datacenter and VPS addresses reports where the server sits, which says nothing about where its operator is.\n\n## L9: other lookups\n\n| Item | Result |\n|---|---|\n| Phone 555-0112 | Landline. Carrier Wrenmoor Telephone. Directory listing: Castellan Hardwood Supply Inc., listed since 1998. |\n| Phone 555-0148 | VoIP. Carrier Callwisp VoIP. No directory listing. |\n| Domain `freemail.example` | Large consumer webmail provider. Anyone can register an address. No lookup was run on the address `dwhitlock.castellan@freemail.example` itself (out of scope). |\n| Public phishing feed | `marrowline-docs.example` reported 24 February 2026. `cstl-docshare.example` reported 12 March 2026 14:30 UTC by a reporter outside the client. `castellan-hardwood.example` and `quennmarsh-accounts.example` not listed. |\n",
"logs/mail-trace.csv": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks mail gateway (gw01) message trace export, 2026-03-01 to 2026-03-17.\n# Filter: sender, recipient or subject contains \"castellan\", \"jpike\" or \"mbeaulieu\". Times are UTC.\nreceived_utc,direction,message_id,sender,recipients,subject,sender_ip,spf,dkim,dmarc,verdict,delivery_action,attachment_name,attachment_sha256\n2026-03-02T15:12:40Z,inbound,<n.8812.20260302@lists.sawdustjournal.example>,news@sawdustjournal.example,jpike@orrinvalley.example,[EXTERNAL] Sawdust Journal - March issue,198.51.100.90,pass,pass,pass,clean,delivered,,\n2026-03-03T14:47:04Z,inbound,<20260303144702.GA11873@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] Invoice CHS-20417,198.51.100.180,pass,pass,pass,clean,delivered,CHS-20417.pdf,a3c91f0e7b24d85c6e10f9a2b7d43c8e5f6a1b09d2e7c4f38a5b6d0e1f2c7a934\n2026-03-04T19:30:11Z,outbound,<7c21e0a9f4b3@mbx01.orrinvalley.example>,jpike@orrinvalley.example,orders@castellanhardwood.example,PO 7744 - walnut 8/4,192.0.2.25,,,,,relayed,PO-7744.pdf,e8b04d2c9a71f35e6c0b8d49a2f1e7c35d6a90b1c4e2f87a3d5b9c0e6f1a2d48\n2026-03-09T13:40:22Z,internal,<3e9b17c0d5a2@mbx01.orrinvalley.example>,mbeaulieu@orrinvalley.example,jpike@orrinvalley.example,Friday run list,,,,,,delivered,,\n2026-03-10T14:02:37Z,inbound,<20260310140231.2C4A1E03@mail.castellan-hardwood.example>,dana.whitlock@castellan-hardwood.example,jpike@orrinvalley.example,[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details,203.0.113.47,pass,none,none,clean,delivered,,\n2026-03-11T13:21:06Z,outbound,<5b2e90d14c7a@mbx01.orrinvalley.example>,jpike@orrinvalley.example,orders@castellanhardwood.example,PO 7751 - hard maple 5/4,192.0.2.25,,,,,relayed,PO-7751.pdf,0f7d3a9c2e15b84d6a09c1e7f3b25d80a4c6e9f1b7d02a53c8e4f6b19d0a7c25\n2026-03-11T13:52:45Z,internal,<a81f3c07e2d94b6e@mbx01.orrinvalley.example>,jpike@orrinvalley.example,mbeaulieu@orrinvalley.example,Castellan banking update - vendor change for Friday run,,,,,,delivered,Castellan_Bank_Letter_2026.pdf,5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b\n2026-03-11T15:06:13Z,internal,<c40d8e2b71f6a953@mbx01.orrinvalley.example>,mbeaulieu@orrinvalley.example,jpike@orrinvalley.example,RE: Castellan banking update - vendor change for Friday run,,,,,,delivered,,\n2026-03-12T16:05:40Z,inbound,<20260312160538.GB20417@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] Reminder: CHS-20417 due 13 March,198.51.100.180,pass,pass,pass,clean,delivered,,\n2026-03-12T16:40:03Z,inbound,<20260312163958.8E71B2C4@mail.castellan-hardwood.example>,dana.whitlock@castellan-hardwood.example,jpike@orrinvalley.example,[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details,203.0.113.47,pass,none,none,clean,delivered,,\n2026-03-13T19:02:09Z,outbound,<erp.000731.20260313@orrinvalley.example>,erp-noreply@orrinvalley.example,ar@castellan-hardwood.example,Remittance advice - Orrin Valley Millworks - payment 000731,192.0.2.25,,,,,relayed,,\n2026-03-16T14:20:55Z,inbound,<20260316142051.GC20417@mail.castellanhardwood.example>,dwhitlock@castellanhardwood.example,ap@orrinvalley.example;jpike@orrinvalley.example,[EXTERNAL] CHS-20417 not received - please call me,198.51.100.180,pass,pass,pass,clean,delivered,,\n",
"logs/mailbox-audit.csv": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks cloud mailbox and sign-in audit export (simplified format).\n# Scope: all events for jpike and mbeaulieu, 2026-03-09T00:00Z to 2026-03-17T12:00Z; admin actions targeting those mailboxes;\n# plus failed sign-ins tenant-wide from 203.0.113.200. Times are UTC. Exported 2026-03-17 by the IT contractor.\ntimestamp_utc,actor,target_mailbox,operation,result,client_ip,user_agent,session_id,details\n2026-03-09T12:58:10Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3a9e41,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-09T13:02:31Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-12c9d0,AuthMethod=Password; MFA=NotRegistered\n2026-03-09T21:47:02Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58b2e7,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T03:11:40Z,jpike,jpike,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:11:52Z,mbeaulieu,mbeaulieu,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:05Z,otran,otran,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:19Z,panand,panand,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:12:33Z,reception,reception,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T03:13:05Z,sales,sales,UserLoginFailed,InvalidPassword,203.0.113.200,python-requests/2.31.0,,AuthMethod=Password\n2026-03-10T13:05:14Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-12f4a8,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T13:08:40Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3b07c2,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-10T14:31:12Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AuthMethod=Password; MFA=NotRegistered\n2026-03-10T14:36:50Z,jpike,jpike,New-InboxRule,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,Name=.; SubjectOrBodyContainsWords=Castellan;remittance;banking;CHS-; MoveToFolder=RSS Subscriptions; MarkAsRead=True; StopProcessingRules=True\n2026-03-10T14:38:02Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Sync; Folder=Inbox; ItemCount=212\n2026-03-10T14:41:27Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Bind; Folder=Inbox; Subject=[EXTERNAL] Invoice CHS-20417\n2026-03-10T14:44:09Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f3a61,AccessType=Sync; Folder=Sent Items; ItemCount=388\n2026-03-10T21:05:36Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58c011,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T12:58:33Z,jpike,jpike,UserLoggedIn,Success,198.51.100.77,\"Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148\",s-58c7f2,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:09:44Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1301b5,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:12:26Z,mbeaulieu,mbeaulieu,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n2026-03-11T13:21:05Z,jpike,jpike,Send,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1301b5,To=orders@castellanhardwood.example; Subject=PO 7751 - hard maple 5/4; Attachments=PO-7751.pdf\n2026-03-11T13:47:20Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,AuthMethod=Password; MFA=NotRegistered\n2026-03-11T13:52:44Z,jpike,jpike,Send,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,To=mbeaulieu@orrinvalley.example; Subject=Castellan banking update - vendor change for Friday run; Attachments=Castellan_Bank_Letter_2026.pdf\n2026-03-11T13:53:05Z,jpike,jpike,MoveToDeletedItems,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,Folder=Sent Items; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T13:53:19Z,jpike,jpike,SoftDelete,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,Folder=Deleted Items; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T14:02:51Z,mbeaulieu,mbeaulieu,MailItemsAccessed,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,AccessType=Bind; Folder=Inbox; Subject=Castellan banking update - vendor change for Friday run\n2026-03-11T15:06:12Z,mbeaulieu,mbeaulieu,Send,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-3c1d90,To=jpike@orrinvalley.example; Subject=RE: Castellan banking update - vendor change for Friday run\n2026-03-11T15:20:31Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-7f9b04,AccessType=Bind; Folder=RSS Subscriptions; Subject=RE: Castellan banking update - vendor change for Friday run\n2026-03-12T13:01:17Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-1377e3,AuthMethod=Password; MFA=NotRegistered\n2026-03-12T16:44:30Z,jpike,jpike,UserLoggedIn,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AuthMethod=Password; MFA=NotRegistered\n2026-03-12T16:45:02Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AccessType=Bind; Folder=RSS Subscriptions; Subject=[EXTERNAL] Reminder: CHS-20417 due 13 March\n2026-03-12T16:45:40Z,jpike,jpike,MailItemsAccessed,Success,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,s-80a2d5,AccessType=Bind; Folder=RSS Subscriptions; Subject=[EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\n2026-03-13T12:55:48Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-13a5f1,AuthMethod=Password; MFA=NotRegistered\n2026-03-16T13:04:22Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-13d2c8,AuthMethod=Password; MFA=NotRegistered\n2026-03-16T19:32:08Z,itadmin,jpike,ResetPassword,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Target=jpike; ForceChangeAtNextLogin=True\n2026-03-16T19:32:40Z,itadmin,jpike,RevokeSessions,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Target=jpike\n2026-03-16T19:40:55Z,itadmin,jpike,Remove-InboxRule,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-a1e0f3,Mailbox=jpike; Identity=.\n2026-03-16T23:14:07Z,jpike,jpike,UserLoginFailed,InvalidPassword,198.51.100.23,Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0,,AuthMethod=Password\n2026-03-17T11:48:31Z,jpike,jpike,UserLoggedIn,Success,192.0.2.10,\"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36\",s-14f7b2,AuthMethod=Password+AuthenticatorApp; MFA=Satisfied\n",
"logs/web-proxy.log": "# SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737).\n# Orrin Valley Millworks web proxy export. Client OVM-WS-AP02 (Jordan Pike's workstation), 2026-03-10 and 2026-03-11.\n# Timestamps: appliance local time. No UTC offset is recorded in this export.\n# TLS inspection: enabled for this client; full URLs are logged.\n# Fields: date time client user method url status bytes category action\n2026-03-10 09:05:02 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/login 200 14022 Business/Email allowed\n2026-03-10 09:05:13 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/login 302 611 Business/Email allowed\n2026-03-10 09:05:14 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 88410 Business/Email allowed\n2026-03-10 09:31:40 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/payment-runs 200 40317 Business/Internal allowed\n2026-03-10 09:58:12 OVM-WS-AP02 jpike GET https://www.weatherdesk.example/forecast 200 120544 News/Weather allowed\n2026-03-10 10:02:50 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-10 10:18:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 90117 Business/Email allowed\n2026-03-10 10:18:51 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/read?id=AAMkAGQ5ZTc0 200 23875 Business/Email allowed\n2026-03-10 10:19:07 OVM-WS-AP02 jpike GET https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&t=anBpa2U%3D 200 5822 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://cstl-docshare.example/assets/js/vw-portal.min.js 200 31740 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://cstl-docshare.example/assets/css/vw.css 200 4410 Newly-Registered-Domain allowed\n2026-03-10 10:19:08 OVM-WS-AP02 jpike GET https://castellanhardwood.example/img/logo.png 200 18204 Business/Industrial allowed\n2026-03-10 10:19:09 OVM-WS-AP02 jpike GET https://cstl-docshare.example/favicon.ico 200 1150 Newly-Registered-Domain allowed\n2026-03-10 10:20:41 OVM-WS-AP02 jpike POST https://cstl-docshare.example/view/auth 302 388 Newly-Registered-Domain allowed\n2026-03-10 10:20:42 OVM-WS-AP02 jpike GET https://castellanhardwood.example/ 200 52310 Business/Industrial allowed\n2026-03-10 10:20:42 OVM-WS-AP02 jpike GET https://castellanhardwood.example/css/site.css?v=2025.3 200 9120 Business/Industrial allowed\n2026-03-10 10:21:30 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 90544 Business/Email allowed\n2026-03-10 11:02:17 OVM-WS-AP02 jpike GET https://www.sawdustjournal.example/march 200 77310 News/Trade allowed\n2026-03-11 09:09:30 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/login 200 14022 Business/Email allowed\n2026-03-11 09:09:43 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/login 302 611 Business/Email allowed\n2026-03-11 09:09:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 87102 Business/Email allowed\n2026-03-11 09:20:18 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/compose 200 12650 Business/Email allowed\n2026-03-11 09:21:05 OVM-WS-AP02 jpike POST https://mail.orrinvalley.example/mail/send 200 1204 Business/Email allowed\n2026-03-11 09:30:12 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/invoices 200 51288 Business/Internal allowed\n2026-03-11 09:48:55 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-11 09:52:30 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/invoices?page=2 200 49870 Business/Internal allowed\n2026-03-11 09:55:10 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/api/notifications 200 902 Business/Email allowed\n2026-03-11 10:01:44 OVM-WS-AP02 jpike GET https://mail.orrinvalley.example/mail/inbox 200 88003 Business/Email allowed\n2026-03-11 10:14:29 OVM-WS-AP02 jpike GET https://erp.orrinvalley.example/ap/payment-runs 200 40999 Business/Internal allowed\n"
}

In [ ]:
import hashlib                     # SHA-256: a impressão digital de cada arquivo
import urllib.request              # baixar sem instalar nada (requests não é stdlib)

RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-7-capstone-career/case-packet/"   # endereço "cru" dos arquivos no GitHub

def baixar(nome, timeout=10):
    """Plano A: baixa o arquivo do GitHub. Devolve o texto ou None se falhar."""
    try:
        with urllib.request.urlopen(RAW + nome, timeout=timeout) as resp:   # timeout: não travar o Colab
            return resp.read().decode("utf-8")                              # bytes -> texto
    except Exception as erro:                                               # sem rede, 404, proxy...
        print(f"   (rede indisponível para {nome}: {type(erro).__name__}) -> uso a cópia embutida")
        return None

def sha(texto):
    """SHA-256 de um texto em UTF-8 (o mesmo que sha256sum daria no arquivo)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

PACOTE = {}                                   # nome do arquivo -> texto que vamos analisar
for nome, copia in EMBUTIDO.items():
    online = baixar(nome)
    if online is None:                         # plano B
        PACOTE[nome] = copia
        origem = "embutido"
    else:                                      # plano A deu certo: confere com a cópia
        PACOTE[nome] = online
        origem = "GitHub" if sha(online) == sha(copia) else "GitHub (DIFERENTE da cópia embutida!)"
    print(f"{nome:32} {sha(PACOTE[nome])[:16]}…  [{origem}]")

## 1. O cofre: `originals/` e `working/`

O original manda copiar o pacote duas vezes. Aqui fazemos o mesmo, só com `pathlib`: gravamos cada arquivo em `capstone/originals/case-packet/` e em `capstone/working/case-packet/`. Daqui para a frente, **só a cópia de trabalho é lida**.

In [ ]:
from pathlib import Path

RAIZ = Path("capstone")                                   # equivale a ~/capstone do original
ORIG = RAIZ / "originals" / "case-packet"                  # o cofre: nunca editado
WORK = RAIZ / "working" / "case-packet"                    # a cópia de trabalho
NOTES = RAIZ / "notes"                                     # suas notas do caso
for pasta in (ORIG, WORK, NOTES):
    pasta.mkdir(parents=True, exist_ok=True)               # exist_ok: rodar duas vezes não quebra

for nome, texto in PACOTE.items():
    for base in (ORIG, WORK):
        destino = base / nome
        destino.parent.mkdir(parents=True, exist_ok=True)  # cria a subpasta logs/
        # newline="" grava os bytes exatamente como vieram (sem trocar \n por \r\n no Windows)
        with open(destino, "w", encoding="utf-8", newline="") as f:
            f.write(texto)

print("Arquivos no cofre:", len(list(ORIG.rglob("*.*"))))

## 2. O hash de cada original, no formato do `sha256sum`

O formato é `hash  caminho` (dois espaços). Usamos esse formato para que o arquivo gerado aqui possa ser conferido no Dia 89 com `sha256sum -c` no Linux, `shasum -a 256 -c` no macOS ou com a função Python logo abaixo.

In [ ]:
import hashlib

def sha256_arquivo(caminho):
    """Lê em blocos de 64 KiB: funciona igual para um log de 4 KB ou uma imagem de disco de 40 GB."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:                      # "rb": os bytes, não o texto
        for bloco in iter(lambda: f.read(65536), b""):  # lê até o fim do arquivo
            h.update(bloco)
    return h.hexdigest()

linhas = []
for p in sorted(ORIG.rglob("*")):                       # sorted: mesma ordem em qualquer máquina
    if p.is_file():
        rel = p.relative_to(ORIG).as_posix()            # "logs/web-proxy.log", com barra normal
        linhas.append(f"{sha256_arquivo(p)}  {rel}")

ARQ_HASH = NOTES / "evidence.sha256"
ARQ_HASH.write_text("\n".join(linhas) + "\n", encoding="utf-8")
print(ARQ_HASH.read_text())

In [ ]:
def conferir(pasta, arquivo_hash):
    """O equivalente Python de `shasum -a 256 -c`: devolve quantos arquivos falharam."""
    falhas = 0
    for linha in Path(arquivo_hash).read_text(encoding="utf-8").splitlines():
        if not linha.strip():
            continue
        esperado, nome = linha.split("  ", 1)           # dois espaços separam hash e caminho
        alvo = Path(pasta) / nome
        atual = sha256_arquivo(alvo) if alvo.exists() else "AUSENTE"
        ok = (atual == esperado)
        falhas += not ok
        print(f"{nome}: {'OK' if ok else 'FALHOU'}")
    return falhas

print("Conferindo o cofre logo depois de gerar (todas devem dar OK):")
print("falhas:", conferir(ORIG, ARQ_HASH))

### 2.1 A cara de um arquivo alterado

O original pede: edite um caractere de um arquivo em `working/`, calcule o hash e compare com o do original. Vamos trocar **um único caractere** do log do proxy (um `9` por um `8` no horário) e ver o que acontece.

In [ ]:
alvo = WORK / "logs" / "web-proxy.log"
texto = alvo.read_text(encoding="utf-8")
adulterado = texto.replace("09:05:13", "08:05:13", 1)        # muda 1 caractere de um horário
with open(alvo, "w", encoding="utf-8", newline="") as f:   # grava a cópia adulterada
    f.write(adulterado)

print("original :", sha256_arquivo(ORIG / "logs" / "web-proxy.log"))
print("adulterad:", sha256_arquivo(alvo))
print("\nConferindo a cópia de TRABALHO contra o registro do cofre:")
conferir(WORK, ARQ_HASH)

# Desfaz a edição, como o original manda.
with open(alvo, "w", encoding="utf-8", newline="") as f:
    f.write(texto)
print("\nEdição desfeita. Trabalho de novo igual ao original:",
      sha256_arquivo(alvo) == sha256_arquivo(ORIG / "logs" / "web-proxy.log"))

## 3. O gabarito do instrutor: hash sim, leitura não

O `07-instructor-key.md` entra no registro, mas **não é aberto** até o relatório do Dia 89 estar pronto. Este notebook não embute o texto dele, só o SHA-256 que tinha quando foi gerado. A célula abaixo baixa o arquivo (se houver rede), confere o hash e **não imprime nada do conteúdo**.

In [ ]:
import urllib.request

HASH_GABARITO_ESPERADO = "c267870d0f19e1743932f97a299e0548e5e906d2bef6cef9f9c4aaa087ad37f3"   # SHA-256 do 07-instructor-key.md quando o notebook foi gerado

try:
    with urllib.request.urlopen(RAW + "07-instructor-key.md", timeout=10) as r:
        dados = r.read()                                    # bytes; não vamos decodificar nem imprimir
    atual = hashlib.sha256(dados).hexdigest()
    print("Gabarito baixado e NÃO lido.")
    print("hash bate com o registro:", atual == HASH_GABARITO_ESPERADO)
    del dados                                               # descarta da memória: tentação a menos
except Exception as erro:
    print(f"Sem rede ({type(erro).__name__}). Registre o hash esperado no seu registro de evidências:")
    print("P10  07-instructor-key.md ", HASH_GABARITO_ESPERADO[:16], "…  (não aberto)")

## 4. O registro de evidências

Agora o registro com a coluna que mais importa: **Limites conhecidos**. Os limites abaixo vêm das notas de recebimento da carta de contratação (P1, Parte C). Leia, ajuste com as suas palavras e salve.

In [ ]:
LIMITES = {   # ID, arquivo, de quem, descrição, limites conhecidos
    "README.md": ("P0", "ThiefDroppers (pacote)", "Índice do pacote", "Não é evidência do caso; é o mapa dos IDs"),
    "01-engagement-letter.md": ("P1", "Owen Tran, cliente", "Contratação, escopo, depoimentos",
        "Depoimentos são resumos, não transcrições; o relato da Castellan é de ouvir dizer"),
    "02-suspicious-email.eml": ("P2", "Priya Anand, TI", "E-mail de phishing bruto",
        "Uma cópia, da caixa do jpike; cabeçalhos acima do gateway são do atacante e podem ser forjados"),
    "03-related-messages.md": ("P3", "Priya Anand, TI", "Mensagens M1 a M7",
        "Cabeçalhos aparados pela TI; metadados do PDF extraídos por terceiros"),
    "04-vendor-site-archived.md": ("P4", "Priya Anand, TI", "Site real arquivado (nov/2025)",
        "Captura de arquivo da web; um só momento, quatro meses antes"),
    "05-lookalike-site-capture.md": ("P5", "Priya Anand, TI", "Site imitação capturado em 17/mar",
        "Um só momento; capturado pela TI, não por você"),
    "06-lookup-results.md": ("P6", "Priya Anand, TI", "WHOIS, DNS, pDNS, CT, varreduras",
        "Consultas de terceiros em 17/mar; você não pode repeti-las"),
    "logs/mail-trace.csv": ("P7", "Priya Anand, TI", "Trace do gateway de e-mail, UTC",
        "Filtrado por castellan/jpike/mbeaulieu; 1 a 17/mar"),
    "logs/mailbox-audit.csv": ("P8", "Priya Anand, TI", "Auditoria de login e caixa, UTC",
        "Só jpike e mbeaulieu; começa em 9/mar; sem ap@; exportado depois da remediação"),
    "logs/web-proxy.log": ("P9", "Priya Anand, TI", "Proxy da estação de Jordan",
        "Fuso não registrado; uma estação só; dois dias só"),
}

linhas_reg = ["| ID | Arquivo | SHA-256 (16) | Recebido | De quem | Descrição | Limites conhecidos |",
              "|---|---|---|---|---|---|---|"]
for nome, (pid, quem, desc, lim) in LIMITES.items():
    h = sha256_arquivo(ORIG / nome)[:16]
    linhas_reg.append(f"| {pid} | {nome} | {h} | 17 mar 2026 | {quem} | {desc} | {lim} |")
linhas_reg.append(f"| P10 | 07-instructor-key.md | {HASH_GABARITO_ESPERADO[:16]} | 17 mar 2026 | ThiefDroppers | Gabarito | NÃO ABERTO até o Dia 89 |")

(NOTES / "evidence-register.md").write_text("# Registro de evidências\n\n" + "\n".join(linhas_reg) + "\n", encoding="utf-8")
print("\n".join(linhas_reg))

## 5. 🎮 Quiz das regras de engajamento

12 ações: 6 do capstone (Orrin Valley) e 6 da Operação Isca. Para cada uma, responda:

- **A** = Autorizado (está na lista, faça e registre)
- **P** = Proibido (não faça; se o caso precisar, recomende no relatório)
- **Z** = Zona cinzenta (pare e peça ao cliente por escrito)

Quer só ler o gabarito? Mude `MODO_JOGO` para `False`.

In [ ]:
ACOES_RE = [
    ("🇨🇦", "Ler e anotar a cópia de trabalho do log de auditoria (P8)", "A",
     "Parte B: revisar todos os arquivos do pacote, inclusive os três logs."),
    ("🇨🇦", "Abrir cstl-docshare.example numa sandbox 'só para ver a página'", "P",
     "Parte B, fora do escopo: visitar, logar ou enviar algo a site do atacante, 'mesmo só para ver'."),
    ("🇨🇦", "Ligar para 555-0148 para ver quem atende", "P",
     "Fora do escopo: contatar o atacante por telefone ou qualquer canal."),
    ("🇨🇦", "Pesquisar o LinkedIn da Dana Whitlock para ver se ela é real", "P",
     "Fora do escopo: vida pessoal e redes de pessoas nomeadas."),
    ("🇨🇦", "Denunciar cstl-docshare.example ao abuso do registrador", "Z",
     "Exemplo resolvido do original: altera evidência; recomende e deixe o cliente decidir."),
    ("🇨🇦", "Pedir uma captura nova do site de phishing, feita pela TI do cliente", "Z",
     "Exemplo resolvido: pedido por escrito ao cliente, registrado no log de decisões."),
    ("🇧🇷", "Calcular o hash de I1, I2 e I3 e montar o registro do Isca", "A",
     "Custódia das evidências que a cooperativa entregou."),
    ("🇧🇷", "Entrar no painel do IscaKit com a senha que estava no config exposto", "P",
     "Sistema alheio: CP art. 154-A. Documente a exposição e entregue."),
    ("🇧🇷", "Consultar no app do seu banco a chave Pix do laranja L-01", "P",
     "O diretório de chaves é para pagar, não investigar; dados de terceiros correm pelos canais da cooperativa."),
    ("🇧🇷", "Pesquisar no Instagram quem é a pessoa por trás do código L-02", "P",
     "Mira pessoa, não padrão. Identificação é trabalho policial com ordem judicial."),
    ("🇧🇷", "Mandar o relatório final direto para a delegacia, sem passar pela cooperativa", "Z",
     "Quem comunica é a vítima; prepare o pacote e peça a decisão por escrito à Marta e ao Dr. Hélio."),
    ("🇧🇷", "Notificar o provedor de 203.0.113.77 pedindo preservação dos registros", "Z",
     "Útil, mas é ação externa em nome do cliente: peça autorização escrita antes. O pedido formal de guarda cautelar previsto no Marco Civil é da autoridade policial, administrativa ou do MP."),
]

MODO_JOGO = True   # False = só mostra o gabarito

acertos = 0
for i, (bandeira, acao, certo, porque) in enumerate(ACOES_RE, start=1):
    print(f"\n[{i:02d}/{len(ACOES_RE)}] {bandeira}  {acao}")
    if MODO_JOGO:
        resp = input("   A, P ou Z? ").strip().upper()[:1]   # só a primeira letra
        if resp not in ("A", "P", "Z"):                         # resposta inválida conta como erro
            print("   (resposta inválida, conta como erro)")
        acertou = (resp == certo)
        acertos += acertou
        print("   ✅ Acertou!" if acertou else f"   ❌ Era {certo}.")
    print(f"   Gabarito: {certo} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(ACOES_RE)}  (Checkpoint BR: pelo menos 9)")

## 6. ➕ O cofre da Operação Isca

As três evidências fictícias do Isca (I1, I2, I3). O mesmo procedimento: hash, registro, limites conhecidos.

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"I1-escopo-assinado.md": "<!-- FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real. -->\n# Contrato VIGIA-2026-001 — escopo (anexo I)\n\nContratante: Cooperativa Serra Azul (fictícia) · Contratada: Vigia (fictícia)\nAssinado por: Marta Quintão, Gerente de Risco · 2026-10-05 11:00 (BRT, UTC-3)\n\nEscopo: Identificar e documentar a infraestrutura do golpe do 'Pix bloqueado' que se passa pela Serra Azul,\nlevantar o padrão das mensagens e preparar notificações de derrubada.\n\nVedado: contatar suspeitos; acessar sistemas de terceiros; tratar dados de cooperados fora dos canais\ninternos da cooperativa; identificar pessoas (atribuição é atribuição policial).\n\nPonto de contato jurídico: Dr. Hélio Brandão (jurídico da cooperativa).\n",
"I2-sms-ouvidoria.csv": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# Amostras repassadas pela ouvidoria da Serra Azul em 2026-10-05. Horário de Brasília (BRT, UTC-3), como aparece no celular.\nrecebido_brt,remetente,texto\n2026-10-05 07:12,+55 00 00000-0001,\"Serra Azul: sua chave Pix foi bloqueada por seguranca. Regularize em serraazul-pix.example\"\n2026-10-05 07:31,+55 00 00000-0002,\"Serra Azul: sua chave Pix foi bloqueada por seguranca. Regularize em lnk.example/sa7\"\n2026-10-05 07:58,+55 00 00000-0003,\"Serra Azul: sua chave Pix foi bloqueada por seguranca. Regularize em serraazul-pix.example\"\n2026-10-05 08:40,+55 00 00000-0004,\"Serra Azul: sua chave Pix foi bloqueada por seguranca. Regularize em lnk.example/sa7\"\n",
"I3-captura-site-falso.txt": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# Captura de texto de https://serraazul-pix.example/ feita em navegador isolado da Vigia, 2026-10-05 14:02 UTC.\nTITULO: Serra Azul | Regularizacao de Chave Pix\nTEXTO: Sua chave Pix foi bloqueada preventivamente. Para desbloquear, confirme seus dados de acesso.\nFORMULARIO: POST /validar  campos: cpf, senha_ib, celular\nRECURSO: /static/ik.js   (comentario no codigo: \"// isca v2 - pescador\")\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-83/"     # pasta caso-continuo/dados/dia-83 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
ISCA_ORIG = Path("isca") / "originais"
ISCA_ORIG.mkdir(parents=True, exist_ok=True)
for nome, texto in ISCA.items():
    open(ISCA_ORIG / nome, "w", encoding="utf-8", newline="").write(texto)

LIM_ISCA = {
    "I1-escopo-assinado.md": ("I1", "Marta Quintão", "Escopo assinado 05/10 11h00 BRT", "Anexo do contrato; o contrato completo fica com o jurídico"),
    "I2-sms-ouvidoria.csv": ("I2", "Ouvidoria da Serra Azul", "4 amostras de SMS", "Horário BRT como no celular; 4 de 37 reclamações; sem o aparelho original"),
    "I3-captura-site-falso.txt": ("I3", "Vigia (você)", "Captura de texto do site falso", "Texto, não tela; um só momento (14:02 UTC); o site pode ter mudado"),
}
reg_isca = ["| ID | Arquivo | SHA-256 (16) | De quem | Descrição | Limites conhecidos |", "|---|---|---|---|---|---|"]
sums = []
for nome, (iid, quem, desc, lim) in LIM_ISCA.items():
    h = sha256_arquivo(ISCA_ORIG / nome)
    sums.append(f"{h}  {nome}")
    reg_isca.append(f"| {iid} | {nome} | {h[:16]} | {quem} | {desc} | {lim} |")

Path("notas").mkdir(exist_ok=True)
Path("notas/evidencias-isca.sha256").write_text("\n".join(sums) + "\n", encoding="utf-8")
Path("notas/registro-evidencias-isca.md").write_text("# Registro de evidências — Operação Isca\n\n" + "\n".join(reg_isca) + "\n", encoding="utf-8")
print("\n".join(reg_isca))
print("\nConferência imediata:")
conferir(ISCA_ORIG, "notas/evidencias-isca.sha256")

## 7. 📝 Registro de decisões (primeira linha)

Preencha e rode. A data UTC é colocada pelo próprio Python.

In [ ]:
#@title Registro de decisões { display-mode: "form" }
decisao_1 = "Dia 83: hash do pacote calculado, 11 arquivos, todos OK. Não abri o P10."  #@param {type:"string"}
decisao_2 = "Dia 83 (Isca): hash de I1, I2, I3 calculado. Não acesso o painel do IscaKit nem os remetentes."  #@param {type:"string"}
zona_cinzenta = "Relatório para a delegacia: preparo o pacote, quem entrega é a cooperativa. Pedido por escrito à Marta."  #@param {type:"string"}

from datetime import datetime, timezone
agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%MZ")      # hora UTC, como pede o Dia 01
log = "# Registro de decisões\n\n" + "".join(f"- {agora} — {d}\n" for d in (decisao_1, decisao_2, zona_cinzenta))
(NOTES / "decisions-log.md").write_text(log, encoding="utf-8")
print(log)

try:                                                   # no Colab, baixa as notas
    from google.colab import files
    for arq in ("evidence.sha256", "evidence-register.md", "decisions-log.md"):
        files.download(str(NOTES / arq))
    files.download("notas/registro-evidencias-isca.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em capstone/notes/ e notas/)")

## ✅ Antes de fechar o Dia 83

- [ ] `capstone/notes/evidence.sha256` gerado e conferido (todas as linhas OK)
- [ ] Você viu a divergência de um arquivo alterado e desfez a edição
- [ ] Registro de evidências com limites conhecidos em todas as linhas (P10 sem abrir)
- [ ] Quiz com pelo menos 9/12
- [ ] Registro de evidências do Isca (I1–I3) e registro de decisões iniciado

**Amanhã:** Dia 84: reconhecimento da infraestrutura externa. Um e-mail vira um mapa, e cada pivô passa pelo teste da explicação inocente.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>